In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock

In [ ]:
df = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])

In [ ]:
sherlock(df)

In [ ]:
df.head()

---
# CA

code_compte == '70600000' -> loyers

## CA Global

In [ ]:
loyers = df.query("code_compte == '70600000'")

In [ ]:
credit = loyers['crédit_euro'].sum()
debit = loyers['débit_euro'].sum()
credit - debit

## CA Annuel

In [ ]:
loyers.groupby('année')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro')

## CA Mensuel

In [ ]:
loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro')

## CA par Plateforme

In [ ]:
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

### CA par PTF

In [ ]:
loyers.groupby('canal')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro')

### CA par PTF annuel

In [ ]:
loyers.groupby(['année', 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro')

### CA par PTF mensuel

In [ ]:
loyers.groupby([loyers['date_écriture'].dt.to_period('M'), 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro')


---
# Charges

In [ ]:
charges = df[df['code_compte'].str.startswith('6') & (df['débit_euro'] > 0)]
charges.head(0)

## Details par Categories

In [ ]:
charges.groupby(['code_compte', 'intitulé_compte'])['débit_euro'].sum()

## Charges Mensuelles

In [ ]:
charges.groupby(df['date_écriture'].dt.to_period('M'))['débit_euro'].sum()

### Focus sur 12/2025

In [ ]:
charges[(charges['date_écriture'].dt.to_period('M') == '2024-12')]

## Mensuel de la Categorie "Entretien et petit équipement"

In [ ]:
charges.query("code_compte == '60630000'").groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum()

### Focus sur 04/2025

In [ ]:
charges[(charges['code_compte'] == '60630000') & (charges['date_écriture'].dt.to_period('M') == '2025-04')]

## Fixes vs Variables

In [ ]:
fixes = ['61610000', '66116000', '62620000', '62780000', '63512000', '62810000', '68112000']
semi_fixes = ['60630000', '61520000', '61560000']
variables = ['60611000', '60614000', '62220000', '62260000', '62270000', '62310000', '62340000', '62510000', '62610000', '65800000']

conds = [
    charges['code_compte'].isin(fixes),
    charges['code_compte'].isin(semi_fixes),
    charges['code_compte'].isin(variables)
]

choices = ['fixe', 'semi_fixe', 'variable']

charges['categorie'] = np.select(conds, choices, default='autre')

In [ ]:
# Sommes par Catégories

charges.groupby('categorie')['débit_euro'].sum()

# Cashflow

## Mensuel

In [ ]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow

## Annuel

In [ ]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('Y'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('Y'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow